# NBuffer 多缓冲

单缓冲的搬运与计算只能串行：搬入 → 等 → 计算 → 等 → 搬出。多缓冲（NBuffer）用多块 Tile 轮转，让"搬运第 k+1 块"与"计算第 k 块"重叠进行——这是流水线优化的第一块基石。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 为什么需要双缓冲

单缓冲时序（时间向下）：

```text
搬入 tile_0 ── 计算 tile_0 ── 搬入 tile_1 ── 计算 tile_1 ── ...
              ↑ 同一缓冲的生产/消费存在依赖；空闲比例取决于各阶段耗时
```

双缓冲时序（两块 Tile 交替，即 ping-pong——一块在被计算时，另一块在被搬入）：

```text
时间 →    t0        t1            t2            t3
槽 0：   搬入块0    计算块0       搬出块0
槽 1：             搬入块1       计算块1       搬出块1
```

从 t1 起搬运与计算开始重叠；稳态下每块耗时约为 max(T_load, T_compute)，而串行执行是 T_load + T_compute。实际收益还受写回、同步和流水首尾开销影响。槽位只有 0、1 两个，复用前必须确认上一次的消费者（包括 store 搬出）已经完成。


## 2. make_tile_group：创建轮转 Tile

```python
import pypto_pro.language as pl

tt = pl.TileType(shape=[64, 128], dtype=pl.DT_FP16, target_memory=pl.MemorySpace.Vec)

# 双缓冲：mutex_ids 两个 ID，框架自动同步
double_buffer = pl.make_tile_group(type=tt, addrs=0x0000, mutex_ids=[0, 1])
```

配置含义：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">配置</th><th style="text-align:left;">含义</th></tr>
<tr><td><code>mutex_ids=[0]</code></td><td>单块 Tile，无轮转效果</td></tr>
<tr><td><code>mutex_ids=[0, 1]</code></td><td>两块 Tile 交替，双缓冲（ping-pong）</td></tr>
<tr><td><code>mutex_ids=[0, 1, 2, ...]</code></td><td>三缓冲及更深的 N 缓冲</td></tr>
<tr><td><code>mutex_ids=None, depth=N</code></td><td>N 块 Tile 轮转，但不提供自动同步（自行管理时序）</td></tr>
</table>

### 地址配置的两种形式

```python
# 单基地址：各 Tile 按"单槽位大小"连续排布
group = pl.make_tile_group(type=tt, addrs=0x0000, mutex_ids=[0, 1])

# 地址列表：显式指定每块地址
group = pl.make_tile_group(type=tt, addrs=[0x0000, 0x10000], mutex_ids=[2, 3])
```

单基地址形式下，框架按单块 Tile 占用字节数自动计算后续地址（如 FP16 [64,128] 单块 0x4000，则两块落在 0x0000 与 0x4000）。


## 3. 轮转访问：next / current / previous / 下标

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">访问方式</th><th style="text-align:left;">推进轮转位置</th><th style="text-align:left;">说明</th></tr>
<tr><td><code>group.next()</code></td><td>是</td><td>推进并返回下一块 Tile，连续调用循环选择</td></tr>
<tr><td><code>group.current()</code></td><td>否</td><td>返回当前 Tile</td></tr>
<tr><td><code>group.previous()</code></td><td>否</td><td>返回前一块 Tile</td></tr>
<tr><td><code>group[i]</code></td><td>否</td><td>按下标取 Tile（编译期或运行时整数，运行时须在 [0, depth)）</td></tr>
</table>

**初始位置注意**：TileGroup 的初始轮转位置是**最后一块** Tile——第一次调用 `next()` 返回第 0 块；调用 `next()` 之前，`current()` 返回最后一块。

```python
current_tile = double_buffer.current()   # 初始：第 1 块（最后一块）
next_tile = double_buffer.next()         # 推进到第 0 块
first_tile = double_buffer[0]            # 直接取第 0 块，不动游标
```

【待 950 验证：轮转语义示例，取自官方文档】

使用建议：流水迭代中用 `next()` 推进；仅需固定访问某块时用 `current()` 或下标，**避免无意改变轮转位置**。


## 4. mutex ID 的规划原则

mutex ID 是片上互斥同步资源的编号，取值 [0, 31]：

1. **优先给不同 Tile 分配不同 ID**：如双缓冲用 [0, 1]，同时参与流水的另一 Tile 用 [2]——避免无关 Tile 因共用 ID 互相等待；
2. ID 不够用时，让"使用时段不重叠"的 Tile 复用同一 ID；**尽量不要让可能同时使用的两块 Tile 共用**；
3. 本章使用显式 ID 列表，如 `mutex_ids=[0, 1]`；只有安装包支持自动分配时，才可使用 `mutex_ids="auto"`（须指定 depth 且启用 auto_mutex）。该能力不能仅凭版本号 0.3.0 推断，部分配套包仅接受列表、元组或 None；
4. `mutex_ids` 非空时**必须启用** `@pl.jit(auto_mutex=True)`，框架据此自动插入同步。


## 5. 双缓冲搬运计算骨架

```python
@pl.jit(auto_mutex=True)
def copy_double_buffer_kernel(
    x: pl.Tensor[[4, 128], pl.DT_FP16],
    out: pl.Tensor[[4, 128], pl.DT_FP16],
):
    tt = pl.TileType(shape=[2, 128], dtype=pl.DT_FP16, target_memory=pl.MemorySpace.Vec)
    buf = pl.make_tile_group(type=tt, addrs=0x0000, mutex_ids=[0, 1])

    with pl.section_vector():
        for i in pl.range(0, 2):
            tile = buf.next()
            pl.load(tile, x, [i * 2, 0])      # 搬入第 i 块
            pl.store(out, tile, [i * 2, 0])   # 搬出第 i 块
```

【待 950 验证：双缓冲骨架（演示轮转写法，搬入与搬出也可能重叠；实际收益取决于流水调度与同步开销。带计算的例子见 02.08 实践）】


### 带计算的完整例子：预取、计算与收尾

输入为 [8,256] FP32，每次处理 4 行，因此有两个任务。两个槽位各占 0x1000 B，放在 0 和 0x1000。

**为什么进循环前要先搬一块（预热）？** TileGroup 的初始游标停在最后一块，循环里 `current()` 取到的槽位必须已经装好本次要算的数据；不预热，第一轮 `cur` 里就是未初始化的 UB 内容。

循环体内的顺序是关键：先取当前槽位 `cur`，非末轮再 `next()` 预取下一块并搬入，然后计算 `cur` 并写回。对照上一小节的时间轴——第 0 轮预取块 1 的搬运与计算块 0 重叠（t1）；第 1 轮是末轮，不再预取，只计算块 1 并写回。每个任务只 load/store 一次，GM 行起点分别是 0 和 4。


In [ ]:
@pl.jit(auto_mutex=True)
def scale_prefetch(x: pl.Tensor[[8, 256], pl.DT_FP32],
                   out: pl.Tensor[[8, 256], pl.DT_FP32]):
    tt = pl.TileType(shape=[4, 256], dtype=pl.DT_FP32, target_memory=pl.MemorySpace.Vec)
    group = pl.make_tile_group(type=tt, addrs=0, mutex_ids=[0, 1])
    with pl.section_vector():
        # 预热：进稳态循环前先搬入第 0 块
        first = group.next()
        pl.load(first, x, [0, 0])
        for task in pl.range(0, 2):
            cur = group.current()              # 本轮要计算的槽位
            if task + 1 < 2:                   # 非末轮才预取
                nxt = group.next()
                pl.load(nxt, x, [(task + 1) * 4, 0])   # 预取搬运与计算重叠
            pl.mul(cur, cur, 2.0)
            pl.store(out, cur, [task * 4, 0])


In [ ]:
device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
x = torch.arange(2048, device=device, dtype=torch.float32).reshape(8, 256)
out = torch.full_like(x, float('nan'))
scale_prefetch[None, 1](x, out)
torch.npu.synchronize()
assert torch.isfinite(out).all().item()
torch.testing.assert_close(out, x * 2, rtol=1e-5, atol=1e-6)
print('双缓冲缩放校验通过')


## 6. 课后练习

1. （填空）make_tile_group 的初始轮转位置是第 ______ 块 Tile，因此首次调用 next() 返回第 ______ 块。
2. （选择）两块"可能同时参与流水"的 Tile 共用一个 mutex ID，会导致：
   A. 编译报错<br/>
   B. 互相等待、降低流水并行度<br/>
   C. 数据错误
3. （实践）一个三缓冲 TileGroup 从 0x0000 用单基地址排布，TileType 为 [64, 128] FP16——三块 Tile 各落在什么地址？

In [ ]:
!cat ./answer/02.05_answer.txt

## 7. 本节小结

- 多缓冲让**搬运与计算重叠**：双缓冲是最常用的形态；
- `make_tile_group` 统一管理地址、轮转与 mutex 元数据；单基地址自动连续排布；
- `next()` 推进游标（初始位置在最后一块）、`current()/下标` 不推进；
- mutex ID [0,31]：不同 Tile 优先不同 ID，可能与同时用的不共用；配 mutex_ids 必须开 auto_mutex。